# Musicm8 — free GPU training on Kaggle

This notebook trains the **tiny Musicm8 model** on audio you are authorized to use.

> Free GPU availability and session limits can change. Only train on audio you have permission to use.

### Before running
1. Turn on a **GPU accelerator** in Kaggle notebook settings.
2. Add a Kaggle Dataset containing your audio.
3. Enable Internet so the notebook can clone GitHub and download pretrained codec/text weights.
4. If the dataset contains `manifest.jsonl`, it is used automatically; otherwise captions are made from filenames.


In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Enable a GPU accelerator in Kaggle settings and restart the session.')


## 1. Clone Musicm8 and install requirements


In [ ]:
!rm -rf /kaggle/working/Musicm8
!git clone --depth 1 https://github.com/Elephant-logic/Musicm8.git /kaggle/working/Musicm8
%cd /kaggle/working/Musicm8
!python -m pip install -q --upgrade pip
!pip install -q -r requirements.txt


## 2. Find the attached dataset and configure the run


In [ ]:
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/musicm8-work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
candidate_dirs = [p for p in INPUT_ROOT.iterdir() if p.is_dir()]
if not candidate_dirs:
    raise FileNotFoundError('No Kaggle Dataset is attached. Use Add Input / Add Data first.')
AUDIO_DIR = candidate_dirs[0]  # change this if you attached several datasets
print('Using dataset:', AUDIO_DIR)
print('Available datasets:', [p.name for p in candidate_dirs])
CODEC = 'encodec24'
CLIP_SECONDS = 8
STRIDE_SECONDS = 8
STEPS = 2000
BATCH_SIZE = 2
GRAD_ACCUM = 2
SAVE_EVERY = 250
RUN_NAME = 'tiny-overfit'


## 3. Find or build the manifest


In [ ]:
import json
found = list(AUDIO_DIR.rglob('manifest.jsonl'))
if found:
    MANIFEST = found[0]
    print('Using manifest:', MANIFEST)
else:
    MANIFEST = WORK_DIR / 'manifest.jsonl'
    exts = {'.wav','.mp3','.flac','.m4a','.ogg','.aac'}
    files = sorted(p for p in AUDIO_DIR.rglob('*') if p.suffix.lower() in exts)
    if not files:
        raise FileNotFoundError(f'No audio files found under {AUDIO_DIR}')
    with MANIFEST.open('w', encoding='utf-8') as f:
        for p in files:
            caption = p.stem.replace('_',' ').replace('-',' ')
            f.write(json.dumps({'audio': str(p), 'caption': f'music track, {caption}'}, ensure_ascii=False) + '\n')
    print('Created manifest with', len(files), 'tracks')
print(MANIFEST.read_text(encoding='utf-8')[:1500])


## 4. Tokenize audio


In [ ]:
TOKENS_DIR = WORK_DIR / f'tokens-{CODEC}'
INDEX = TOKENS_DIR / 'index.jsonl'
if not INDEX.exists():
    cmd = [sys.executable, 'tokenize_dataset.py', '--manifest', str(MANIFEST), '--out', str(TOKENS_DIR), '--codec', CODEC, '--channels', '1', '--clip-seconds', str(CLIP_SECONDS), '--stride-seconds', str(STRIDE_SECONDS), '--keep-tail', '--device', 'cuda']
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Using cached tokens:', INDEX)


## 5. Train / resume
Kaggle `/kaggle/working` lasts for the current session. Use **Save Version** to preserve outputs, then attach a previous output/checkpoint to resume another day.


In [ ]:
RUN_DIR = WORK_DIR / 'runs' / RUN_NAME
LATEST = RUN_DIR / 'latest.pt'
cmd = [sys.executable, 'train.py', '--data', str(INDEX), '--config', 'configs/v2-tiny.json', '--out', str(RUN_DIR), '--batch-size', str(BATCH_SIZE), '--grad-accum', str(GRAD_ACCUM), '--steps', str(STEPS), '--save-every', str(SAVE_EVERY), '--num-workers', '2', '--device', 'cuda']
if LATEST.exists():
    cmd += ['--resume', str(LATEST)]
    print('Resuming:', LATEST)
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Checkpoint:', LATEST)


## 6. Generate a sample and bundle outputs


In [ ]:
SAMPLE = WORK_DIR / 'sample.wav'
PROMPT = 'dark atmospheric electronic music with deep bass and wide synth pads'
subprocess.run([sys.executable, 'generate.py', '--checkpoint', str(LATEST), '--prompt', PROMPT, '--seconds', '8', '--seed', '42', '--out', str(SAMPLE)], check=True)
from IPython.display import Audio, display
display(Audio(str(SAMPLE)))
import shutil
archive = shutil.make_archive('/kaggle/working/musicm8-output', 'zip', WORK_DIR)
print('Saved output bundle:', archive)
print('Use Kaggle Save Version / Output to keep or download the checkpoint and WAV.')


## Resume another day
Save `latest.pt` as a Kaggle output/dataset, attach it to a later session, copy it into `WORK_DIR/runs/tiny-overfit/latest.pt`, and rerun training with a larger `STEPS` value.
